In [ ]:
from google.colab import drive
drive.mount("<mount-point>")

In [ ]:
# !pip install bitsandbytes
# !pip install datasets

import numpy as np
import pandas as pd
import re

import json
from transformers import pipeline
import torch
# from datasets import load_dataset, Dataset

## Things to edit

In [ ]:
# 9

test_path = "<project-data-path>"   # Testing data
output_path = "<project-data-path>" # Output path
model_path = "<project-data-path>"  # Model path, EDIT before running all

test_data = pd.read_json(f'{test_path}instruct_test_fold_9.json', lines=True) # EDIT before running all

In [ ]:
test_data['instruction'][0]

## Preparing your model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Remember to modify the following API to yours.

ChatAnywhere: https://chatanywhere.apifox.cn

In [ ]:
import os
from openai import OpenAI, AzureOpenAI

# CityU client
cityu_client = AzureOpenAI(
  azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT", "https://<your-azure-openai-endpoint>"),
  api_key=os.environ["AZURE_OPENAI_API_KEY"],
  api_version="2024-02-01"
)

# Personal client
chat_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.chatanywhere.tech/v1")
)

In [ ]:
from transformers import AutoConfig,AutoTokenizer,AutoModelForCausalLM,pipeline
import json
import os

tokenizer = AutoTokenizer.from_pretrained(model_path)

pipe = pipeline(
    "text-generation",
    model=model_path,
    # The quantization line
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

terminators = [
    pipe.tokenizer.eos_token_id,
    pipe.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

## Preparing your data

In [ ]:
from tqdm import tqdm

Special Notes for inference:
- Do not set the `max_new_tokens` too large (200, or even 150 is quite enough), which may instruct the model to generate irrelevant things
- Chat template is not available on my part, if it is available for you, you can have a try to see whether it increase controllability.

In [ ]:
def get_validity_pred_label(
    article, question, distractor
    ):

    ending = "Indicate only the letter T or F."

    validity_prompt = f"""
        Text:
        {article}

        Question: {question}

        Answer: {distractor}

        Based on the text above, is this answer correct (T) or incorrect (F)? {ending}
    """
    validity_prompt = validity_prompt.replace("\n        ","\n").lstrip().rstrip()

    try:
        response = cityu_client.chat.completions.create(
        model="gpt-4o-ca",
        messages=[
            {"role": "user", "content": validity_prompt}
            ],
        max_tokens=250,
        temperature=0.6,
        top_p=0.9
        )
    except:
        response = chat_client.chat.completions.create(
        model="gpt-4o-ca",
        messages=[
            {"role": "user", "content": validity_prompt}
            ],
        max_tokens=250,
        temperature=0.6,
        top_p=0.9
        )

    output = response.choices[0].message.content.lstrip().rstrip().replace('\n', ' ')
    if output.find(f'Final answer:') != -1:
        pred_label = output[output.find(f'Final answer:'):]
        pred_label = pred_label.replace('Final answer:', '').strip()
        pred_label = re.sub(r'[^\w\s]', '', pred_label)
    else:
        pred_label = output.split()[-1]
        pred_label = re.sub(r'[^\w\s]', '', pred_label)

    if pred_label == 'True':
        pred_label = 'T'
    elif pred_label == 'False':
        pred_label = 'F'
    else:
        pred_label = pred_label

    return output, pred_label

In [ ]:
def get_distractor(instruction):
  # chat = [
  #     {"role": "system", "content": ""},
  #     {"role": "user", "content": f'''{instruction}'''}
  # ]
  # chat = f"""
  # <|begin_of_text|><|start_header_id|>system<|end_header_id|><|eot_id|>
  # <|start_header_id|>user<|end_header_id|>{instruction}<|eot_id|>
  # <|start_header_id|>assistant<|end_header_id|>
  # """

  responses = pipe(
      instruction,
      max_new_tokens=150,
      eos_token_id=terminators,
      do_sample=True,
      temperature=0.6,
      top_p=0.9,
      return_full_text=False
    )

  predict_result = responses[0]['generated_text']

  critical_span_match = re.search(
      r'\n\ncritical span[^:]*:\s*(.*?)(?=\n\n|$)',
      predict_result,
      re.IGNORECASE | re.DOTALL
  )
  critical_span = critical_span_match.group(1).strip() if critical_span_match else ""

  distractor_match = re.search(
      r'\n\ndistractor[^:]*:\s*(.*?)(?=\n\n|$)',
      predict_result,
      re.IGNORECASE | re.DOTALL
  )
  distractor = distractor_match.group(1).strip() if distractor_match else ""

  # distractor = predict_result[predict_result.lower().find('distractor:'):]
  # distractor = distractor.replace('Distractor:', '').lstrip().rstrip()

  # critical_span = predict_result[predict_result.lower().find(f'critical span'):]
  # critical_span = critical_span[:critical_span.lower().find('distractor:')]
  # critical_span = critical_span.replace('Critical Span:', '').lstrip().rstrip()

  return predict_result, critical_span, distractor

def batch_get_distractor(prompts, batch_size=8):
  """Get distractor in batch"""
  results = []

  for i in tqdm(range(0, len(prompts), batch_size), desc="Processing"):
      batch_prompts = prompts[i:i + batch_size]

      batch_outputs = pipe(
          batch_prompts,
          max_new_tokens=200,
          eos_token_id=terminators,
          do_sample=True,
          temperature=0.6,
          top_p=0.9,
          pad_token_id=pipe.tokenizer.eos_token_id,
          batch_size=batch_size,
          return_full_text=False
      )

      for output in batch_outputs:
          generated_text = output[0]['generated_text']

          parts = generated_text.split("\n\n")
          critical_span = ""
          distractor = ""

          for part in parts:
            critical_match = re.search(r'critical span[^:]*:\s*(.*)', part, re.IGNORECASE)
            if critical_match:
                critical_span = critical_match.group(1).strip()

            distractor_match = re.search(r'distractor[^:]*:\s*(.*)', part, re.IGNORECASE)
            if distractor_match:
                distractor = distractor_match.group(1).strip()

          results.append({
              "output": generated_text,
              "critical_span": critical_span,
              "distractor": distractor
          })

  return results

In [ ]:
results = get_distractor(test_data['instruction'].values[1])
results

**Note:**

The model output can be messy some time. When deploying it on the website, please remember to check whether the given extractor regex is effective enough to get the tidy distractor.

In [ ]:
# results = batch_get_distractor(test_data['instruction'].iloc[:16].tolist(), batch_size=8)
# results

In [ ]:
from collections import defaultdict

def get_distractor_valid(instruction, max_retries=10):
  passage = instruction[instruction.find("Passage: "):]
  passage = passage[:passage.find("Question: ")]
  passage = passage.replace("Passage: ",'').lstrip().rstrip()

  question = instruction[instruction.find("Question: "):]
  question = question[:question.find("Answer: ")]
  question = question.replace("Question: ",'').lstrip().rstrip()

  validity = 'T'
  count = 0
  result = defaultdict(dict)
  while validity == 'T' and count < max_retries:
    predict_result, critical_span, distractor = get_distractor(instruction)
    _,validity = get_validity_pred_label(
      passage, question, distractor
      )
    result[count] = {
        "output": predict_result,
        "source_article": passage,
        "source_question": question,
        "critical_span": critical_span,
        "distractor": distractor,
        "validity": validity,
        "temperature": 0.6
    }
    count+=1
  return result

def batch_get_distractor_valid(prompts, batch_size=8, max_retries=10):
    """Get valid distractors with two-level indexing structure"""
    batch_results = defaultdict(dict)  # First index: original_idx, Second index: attempt_num

    for i in tqdm(range(0, len(prompts), batch_size), desc="Processing"):
        batch_prompts = prompts[i:i + batch_size]

        # Get article and question
        source_contents = []
        for prompt in batch_prompts:
            article_match = re.search(r'Passage:\s*(.*?)(?=\nQuestion:|$)', prompt, re.DOTALL)
            question_match = re.search(r'Question:\s*(.*?)(?=\nAnswer:|$)', prompt, re.DOTALL)
            source_contents.append({
                "article": article_match.group(1).strip() if article_match else "",
                "question": question_match.group(1).strip() if question_match else ""
            })

        current_batch = batch_prompts.copy()
        retry_indices = list(range(len(current_batch)))
        attempt_counts = [0] * len(current_batch)

        while retry_indices:
            processing_indices = retry_indices.copy()
            retry_indices = []

            batch_outputs = pipe(
                [current_batch[idx] for idx in processing_indices],
                max_new_tokens=200,
                eos_token_id=terminators,
                do_sample=True,
                temperature=0.6,
                top_p=0.9,
                pad_token_id=pipe.tokenizer.eos_token_id,
                batch_size=len(processing_indices),
                return_full_text=False
            )

            for inner_idx, output in enumerate(batch_outputs):
                original_idx = processing_indices[inner_idx]  # Index in the batch
                global_idx = i + original_idx  # Index in the global dataset
                attempt_num = attempt_counts[original_idx] + 1
                generated_text = output[0]['generated_text']

                parts = generated_text.split("\n\n")
                critical_span = ""
                distractor = ""

                for part in parts:
                    critical_match = re.search(r'critical span[^:]*:\s*(.*)', part, re.IGNORECASE)
                    if critical_match:
                        critical_span = critical_match.group(1).strip()

                    distractor_match = re.search(r'distractor[^:]*:\s*(.*)', part, re.IGNORECASE)
                    if distractor_match:
                        distractor = distractor_match.group(1).strip()

                # Validate
                source = source_contents[original_idx]
                if distractor and source["article"] and source["question"]:
                    _, validity = get_validity_pred_label(
                        article=source["article"],
                        question=source["question"],
                        distractor=distractor
                    )
                else:
                    validity = 'T'

                # Record the result
                batch_results[global_idx][attempt_num] = {
                    "output": generated_text,
                    "source_article": source["article"],
                    "source_question": source["question"],
                    "critical_span": critical_span,
                    "distractor": distractor,
                    "validity": validity,
                    "temperature": 0.6
                }

                # Update the count
                attempt_counts[original_idx] = attempt_num

                # Decide whether to retry
                if validity == 'T' and attempt_num < max_retries:
                    retry_indices.append(original_idx)

    return batch_results

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

def batch_get_distractor_valid(prompts, batch_size=8, max_retries=10, output_file="results.json"):
    """Get distractor in batch and save to .json"""
    output_path = Path(output_file)

    # Initialize or load existed data
    if output_path.exists():
        with open(output_path, "r", encoding='utf-8') as f:
            existing_data = json.load(f)
        batch_results = defaultdict(dict, {int(k): v for k, v in existing_data.items()})
        processed_indices = set(batch_results.keys())
    else:
        batch_results = defaultdict(dict)
        processed_indices = set()

    try:
        for i in tqdm(range(0, len(prompts), batch_size), desc="Processing"):
            # Get batch indices
            batch_indices = range(i, min(i+batch_size, len(prompts)))

            # Skip processed idx
            if all(idx in processed_indices for idx in batch_indices):
                continue

            batch_prompts = [prompts[idx] for idx in batch_indices]

            # Get article and question
            source_contents = []
            for prompt in batch_prompts:
                article_match = re.search(r'Passage:\s*(.*?)(?=\nQuestion:|$)', prompt, re.DOTALL)
                question_match = re.search(r'Question:\s*(.*?)(?=\nAnswer:|$)', prompt, re.DOTALL)
                source_contents.append({
                    "article": article_match.group(1).strip() if article_match else "",
                    "question": question_match.group(1).strip() if question_match else ""
                })

            current_batch = batch_prompts.copy()
            retry_indices = list(range(len(current_batch)))
            attempt_counts = [0] * len(current_batch)

            while retry_indices:
                processing_indices = retry_indices.copy()
                retry_indices = []

                batch_outputs = pipe(
                    [current_batch[idx] for idx in processing_indices],
                    max_new_tokens=200,
                    eos_token_id=terminators,
                    do_sample=True,
                    temperature=0.6,
                    top_p=0.9,
                    pad_token_id=pipe.tokenizer.eos_token_id,
                    batch_size=len(processing_indices),
                    return_full_text=False
                )

                for inner_idx, output in enumerate(batch_outputs):
                    original_idx = processing_indices[inner_idx]
                    global_idx = i + original_idx
                    attempt_num = attempt_counts[original_idx] + 1
                    generated_text = output[0]['generated_text']

                    # Generate
                    parts = generated_text.split("\n\n")
                    critical_span = ""
                    distractor = ""

                    for part in parts:
                        critical_match = re.search(r'critical span[^:]*:\s*(.*)', part, re.IGNORECASE)
                        if critical_match:
                            critical_span = critical_match.group(1).strip()

                        distractor_match = re.search(r'distractor[^:]*:\s*(.*)', part, re.IGNORECASE)
                        if distractor_match:
                            distractor = distractor_match.group(1).strip()

                    # Validate
                    source = source_contents[original_idx]
                    if distractor and source["article"] and source["question"]:
                        _, validity = get_validity_pred_label(
                            article=source["article"],
                            question=source["question"],
                            distractor=distractor
                        )
                    else:
                        validity = 'T'

                    # Record
                    batch_results[global_idx][attempt_num] = {
                        "output": generated_text,
                        "source_article": source["article"],
                        "source_question": source["question"],
                        "critical_span": critical_span,
                        "distractor": distractor,
                        "validity": validity,
                        "temperature": 0.6
                    }

                    attempt_counts[original_idx] = attempt_num

                    # Retry
                    if validity == 'T' and attempt_num < max_retries:
                        retry_indices.append(original_idx)

            processed_indices.update(batch_indices)

            with open(output_path, "w", encoding='utf-8') as f:
                json.dump(dict(batch_results), f, indent=2, ensure_ascii=False)

    except Exception as e:
        print(f"Error, but save to {output_path}")
        raise

    return batch_results

In [ ]:
# results = batch_get_distractor_valid(test_data['instruction'].iloc[:8].tolist(), batch_size=4)
# results

## For all inference

In [ ]:
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(f'Model: {model_path}')
print(f'Output: {output_path}')
print(f"Output file: {output_path + model_path.split('/')[-1]}.json")

In [ ]:
results = defaultdict(dict)
for i,row in tqdm(test_data.iterrows()):
  result = get_distractor_valid(row['instruction'])
  results[i] = result

with open(f"{output_path + model_path.split('/')[-1]}-modified.json", 'w', encoding='utf-8') as file:
  json.dump(results, file, indent=2, ensure_ascii=False)

In [ ]:
# batch_get_distractor_valid(
#     test_data['instruction'].tolist(),
#     output_file=f"{output_path + model_path.split('/')[-1]}.json",
#     batch_size=8)